In [1]:
import sys
print(sys.executable)


c:\Users\Filip\wine_pc\wine-recommender\.venv\Scripts\python.exe


In [2]:
import torch
print(torch.version.cuda)
print(torch.cuda.is_available())

12.6
True


In [3]:
import numpy as np
import pandas as pd
from torch import nn
import os
import torch
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [4]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Torch version: 2.10.0+cu126
CUDA available: True
GPU name: NVIDIA GeForce RTX 2060
Using device: cuda


In [5]:
df = pd.read_csv('data\XWines_Slim_150K_ratings.csv')

C:\Users\Filip\AppData\Local\Temp\ipykernel_9520\4080725611.py:1: DtypeWarning: Columns (0: Vintage) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data\XWines_Slim_150K_ratings.csv')


In [6]:
df

,RatingID,UserID,WineID,Vintage,Rating,Date
0,143,1356810,103471,1950,4.5,2021-11-02 20:52:59
1,199,1173759,111415,1951,5.0,2015-08-20 17:46:26
2,348,1164877,111395,1952,5.0,2020-11-13 05:40:26
3,374,1207665,111433,1953,5.0,2017-05-05 06:44:13
4,834,1075841,111431,1955,5.0,2016-09-14 20:18:38
...,...,...,...,...,...,...
149995,21013438,1000052,111468,N.V.,4.5,2021-12-22 21:03:51
149996,21013467,1180844,111461,N.V.,4.0,2017-04-23 21:07:55
149997,21013494,1218581,113690,N.V.,3.5,2019-04-14 17:45:08
149998,21013505,1106198,111468,N.V.,4.5,2021-07-10 07:00:15


In [7]:
df.dtypes

RatingID      int64
UserID        int64
WineID        int64
Vintage      object
Rating      float64
Date            str
dtype: object

In [8]:
df = df[["Rating", "UserID", "WineID", "Date"]]
df.sort_values("Date", ascending = True, inplace=True)
df.reset_index(inplace=True)

In [9]:
# dane są JUŻ posortowane po Date rosnąco

split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

# trozbicie x y

X_train = train_df[["UserID", "WineID", "Date"]]
y_train = train_df["Rating"]

X_test  = test_df[["UserID", "WineID", "Date"]]
y_test  = test_df["Rating"]


In [10]:
X_train = X_train.copy()
X_test = X_test.copy()


In [11]:
X_train

,UserID,WineID,Date
0,1200775,155628,2012-04-19 20:46:00
1,1088812,167419,2012-04-21 19:04:41
2,1293899,179012,2012-05-15 00:01:18
3,1293899,111458,2012-05-15 00:51:03
4,1293899,111395,2012-05-15 00:54:22
...,...,...,...
119995,1070825,102055,2020-08-25 01:07:39
119996,1139402,179061,2020-08-25 01:28:38
119997,1107564,179021,2020-08-25 01:42:12
119998,1189563,179131,2020-08-25 01:47:56


In [12]:
users = X_train['UserID'].unique()
wines = X_train['WineID'].unique()
user2_id = {u: i for i, u in enumerate(users)}
wine2_id = {u: i for i, u in enumerate(wines)}

In [13]:
X_train['user_idx'] = X_train['UserID'].map(user2_id)
X_train['wine_idx'] = X_train['WineID'].map(wine2_id)

In [14]:
X_train


,UserID,WineID,Date,user_idx,wine_idx
0,1200775,155628,2012-04-19 20:46:00,0,0
1,1088812,167419,2012-04-21 19:04:41,1,1
2,1293899,179012,2012-05-15 00:01:18,2,2
3,1293899,111458,2012-05-15 00:51:03,2,3
4,1293899,111395,2012-05-15 00:54:22,2,4
...,...,...,...,...,...
119995,1070825,102055,2020-08-25 01:07:39,1222,648
119996,1139402,179061,2020-08-25 01:28:38,3026,12
119997,1107564,179021,2020-08-25 01:42:12,5616,15
119998,1189563,179131,2020-08-25 01:47:56,9361,174


In [15]:
# X_test['user_idx'] = X_test['UserID'].map(user2_id)
# X_test['wine_idx'] = X_test['WineID'].map(wine2_id)

# user_indices =  torch.tensor(X_test['user_idx'])
# wine_indices = torch.tensor(X_test['wine_idx'])

In [16]:
# X_train.copy()
# y_train.copy()

# X_train = X_train.dropna()
# y_train = y_train.loc[X_train.index]

# X_train['user_idx'] = X_train['user_idx'].astype(int)
# X_train['wine_idx'] = X_train['wine_idx'].astype(int)

# X_test_clean['user_idx'] = X_test_clean['user_idx'].astype(int)
# X_test_clean['wine_idx'] = X_test_clean['wine_idx'].astype(int)

# X_user_train = torch.tensor(X_train['user_idx'].values, dtype=torch.long)
# X_wine_train = torch.tensor(X_train['wine_idx'].values, dtype=torch.long)
# y_train_t = torch.tensor(y_train.values, dtype=torch.float32)

# X_user_test = torch.tensor(X_test_clean['user_idx'].values, dtype=torch.long)
# X_wine_test = torch.tensor(X_test_clean['wine_idx'].values, dtype=torch.long)
# y_test_t = torch.tensor(y_test_clean.values, dtype=torch.float32)

In [17]:
#map

X_test = X_test.copy()

X_test['user_idx'] = X_test['UserID'].map(user2_id)
X_test['wine_idx'] = X_test['WineID'].map(wine2_id)

#usuwanie cold start

mask = X_test['user_idx'].notna() & X_test['wine_idx'].notna()

X_test_clean = X_test.loc[mask].copy()
y_test_clean = y_test.loc[X_test_clean.index].copy()


#usuniecie null 

X_train = X_train.copy()
y_train = y_train.copy()

X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]



X_train['user_idx'] = X_train['user_idx'].astype(int)
X_train['wine_idx'] = X_train['wine_idx'].astype(int)

X_test_clean['user_idx'] = X_test_clean['user_idx'].astype(int)
X_test_clean['wine_idx'] = X_test_clean['wine_idx'].astype(int)


# konwersja do tensorów

X_user_train = torch.tensor(
    X_train['user_idx'].values,
    dtype=torch.long
)

X_wine_train = torch.tensor(
    X_train['wine_idx'].values,
    dtype=torch.long
)

y_train_t = torch.tensor(
    y_train.values,
    dtype=torch.float32
)


X_user_test = torch.tensor(
    X_test_clean['user_idx'].values,
    dtype=torch.long
)

X_wine_test = torch.tensor(
    X_test_clean['wine_idx'].values,
    dtype=torch.long
)

y_test_t = torch.tensor(
    y_test_clean.values,
    dtype=torch.float32
)


In [18]:
num_users = len(user2_id)
num_wines = len(wine2_id)

In [19]:
print(X_test.isnull().sum())

UserID         0
WineID         0
Date           0
user_idx    1782
wine_idx      22
dtype: int64


In [20]:
mask = X_test['user_idx'].notna() & X_test['wine_idx'].notna()

X_test_clean = X_test[mask]
y_test_clean = y_test[mask]

In [21]:
y_test_clean

120000    4.0
120001    4.5
120002    4.0
120003    3.0
120004    4.0
         ... 
149995    3.5
149996    4.0
149997    4.0
149998    4.5
149999    2.0
Name: Rating, Length: 28200, dtype: float64

In [22]:
print(y_test_clean.isnull().sum())

0


In [23]:
# funcke do metryk

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

def calculate_regression_metrics(y_true, y_pred):
    """Oblicza podstawowe metryki regresji"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2}

def precision_at_k(y_true, y_pred, k=10, threshold=4.0):

    # Precision@K - ile z top-K rekomendacji to trafne rekomendacje
    # threshold - ocena powyżej której wino jest uważane za 'dobre'

    sorted_indices = np.argsort(y_pred)[::-1][:k]
    relevant = np.sum(y_true[sorted_indices] >= threshold)
    return relevant / k

def recall_at_k(y_true, y_pred, k=10, threshold=4.0):

    # Recall@K - jaka część wszystkich dobrych win została znaleziona w top-K

    sorted_indices = np.argsort(y_pred)[::-1][:k]
    relevant_in_topk = np.sum(y_true[sorted_indices] >= threshold)
    total_relevant = np.sum(y_true >= threshold)
    if total_relevant == 0:
        return 0
    return relevant_in_topk / total_relevant

def ndcg_at_k(y_true, y_pred, k=10):

    # NDCG@K (Normalized Discounted Cumulative Gain)
    # Mierzy jakość rankingu uwzględniając pozycję elementu

    def dcg_at_k(r, k):
        r = np.asarray(r, dtype=np.float64)[:k] 
        if r.size:
            return np.sum(r / np.log2(np.arange(2, r.size + 2)))
        return 0.
    
    sorted_indices = np.argsort(y_pred)[::-1][:k]
    actual_dcg = dcg_at_k(y_true[sorted_indices], k)
    
    ideal_indices = np.argsort(y_true)[::-1][:k]
    ideal_dcg = dcg_at_k(y_true[ideal_indices], k)
    
    if ideal_dcg == 0:
        return 0
    return actual_dcg / ideal_dcg
    
def calculate_ranking_metrics_per_user(X_test_df, y_test_array, y_pred_array, k=10):
  
    # Oblicza metryki rankingowe dla każdego użytkownika osobno, potem uśrednia

    X_test_df = X_test_df.copy()
    X_test_df['y_true'] = y_test_array
    X_test_df['y_pred'] = y_pred_array
    
    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    
    for user_id in X_test_df['user_idx'].unique():
        user_data = X_test_df[X_test_df['user_idx'] == user_id]
        if len(user_data) < k:
            continue
            
        y_true_user = user_data['y_true'].values
        y_pred_user = user_data['y_pred'].values
        
        precision_scores.append(precision_at_k(y_true_user, y_pred_user, k))
        recall_scores.append(recall_at_k(y_true_user, y_pred_user, k))
        ndcg_scores.append(ndcg_at_k(y_true_user, y_pred_user, k))
    
    return {
        f'Precision@{k}': np.mean(precision_scores),
        f'Recall@{k}': np.mean(recall_scores),
        f'NDCG@{k}': np.mean(ndcg_scores)
    }

def calculate_coverage(model, user2_id, wine2_id, device, threshold=3.5):

    # Catalog Coverage - jaki procent win model potrafi rekomendować

    model.eval()
    recommended_wines = set()
    
    with torch.no_grad():
        sample_users = np.random.choice(list(user2_id.values()), 
                                       min(100, len(user2_id)), 
                                       replace=False)
        
        for user_idx in sample_users:
            users_tensor = torch.tensor([user_idx] * len(wine2_id), 
                                       dtype=torch.long, device=device)
            wines_tensor = torch.tensor(list(wine2_id.values()), 
                                       dtype=torch.long, device=device)
            
            predictions = model(users_tensor, wines_tensor).squeeze().cpu().numpy()
            
            recommended = np.where(predictions >= threshold)[0]
            recommended_wines.update(recommended)
    
    coverage = len(recommended_wines) / len(wine2_id)
    return coverage

def evaluate_model(model, test_loader, X_test_df, y_test_array, 
                  user2_id, wine2_id, device, model_name):

    # Kompleksowa ewaluacja modelu

    model.eval()
    all_preds = []
    all_true = []
    
    with torch.no_grad():
        for user, wine, rating in test_loader:
            user = user.to(device)
            wine = wine.to(device)
            preds = model(user, wine).squeeze().cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(rating.numpy())
    
    all_preds = np.array(all_preds)
    all_true = np.array(all_true)
    
   
    regression_metrics = calculate_regression_metrics(all_true, all_preds)
    
   
    ranking_metrics_10 = calculate_ranking_metrics_per_user(
        X_test_df, all_true, all_preds, k=10
    )
    ranking_metrics_5 = calculate_ranking_metrics_per_user(
        X_test_df, all_true, all_preds, k=5
    )
    
 
    coverage = calculate_coverage(model, user2_id, wine2_id, device)
    
   
    all_metrics = {
        'Model': model_name,
        **regression_metrics,
        **ranking_metrics_5,
        **ranking_metrics_10,
        'Coverage': coverage
    }
    
    return all_metrics, all_preds

## Rp3beta 



In [34]:
from sklearn.model_selection import train_test_split

def train_test_split_per_user(df, test_size=0.2, random_state=42):
    train_list = []
    test_list = []

    for user_id, user_data in df.groupby('UserID'):
        if len(user_data) < 2:
            continue  # pomijamy użytkowników z 1 oceną
        
        train_u, test_u = train_test_split(
            user_data,
            test_size=test_size,
            random_state=random_state
        )
        
        train_list.append(train_u)
        test_list.append(test_u)

    train_df = pd.concat(train_list)
    test_df = pd.concat(test_list)

    return train_df, test_df

In [32]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

class RP3Beta:
    def __init__(self, beta = 0.6, top_k = 100):
        self.beta = beta
        self.top_k = top_k
        self.user_item_matrix = None
        self.item_scores = None
    
    def fit(self,df, user_col = 'UserID', item_col = 'WineID', rating_col = 'Rating'):
        self.users = df[user_col].unique()
        self.items = df[item_col].unique()
        self.user2idx = {u: i for i, u in enumerate(self.users)}
        self.item2idx = {i: j for j, i in enumerate(self.items)}

        #sparsity matrix

        rows = df[user_col].map(self.user2idx).values
        cols = df[item_col].map(self.item2idx).values
        data = df[rating_col].values
        self.user_item_matrix = csr_matrix((data,(rows,cols)), shape= (len(self.users), len(self.items)))

        #dodanie bety
        self.item_degree = np.array(self.user_item_matrix.sum(axis=0)).flatten() ** self.beta

    def recommend(self, user_id, top_n = None):
        if top_n is None:
            top_n = self.top_k

        u_idx = self.user2idx.get(user_id)
        if u_idx is None:
            return []
        
        user_row = self.user_item_matrix[u_idx].toarray().flatten()

        item_item_matrix = self.user_item_matrix.T @ self.user_item_matrix

        scores  = item_item_matrix @ user_row
        scores = scores / self.item_degree

        scores[user_row > 0] = -np.inf 

        top_indices = np.argsort(scores)[::-1][:top_n]
        top_items = [self.items[i] for i in top_indices]
        top_scores = scores[top_indices]

        return list(zip(top_items, top_scores))

In [35]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k


def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return 1 if len(set(recommended_k) & set(relevant)) > 0 else 0

In [36]:
def evaluate_rp3beta(model, train_df, test_df, k=10):

    precisions = []
    recalls = []
    hit_rates = []

    test_users = test_df['UserID'].unique()

    for user in test_users:

        # wina w teście (ground truth)
        relevant_items = test_df[test_df['UserID'] == user]['WineID'].tolist()

        # jeśli użytkownika nie było w train → pomijamy
        if user not in model.user2idx:
            continue

        # rekomendacje
        recommendations = model.recommend(user, top_n=k)
        recommended_items = [item for item, score in recommendations]

        # liczymy metryki
        precisions.append(precision_at_k(recommended_items, relevant_items, k))
        recalls.append(recall_at_k(recommended_items, relevant_items, k))
        hit_rates.append(hit_rate_at_k(recommended_items, relevant_items, k))

    return {
        "Precision@{}".format(k): np.mean(precisions),
        "Recall@{}".format(k): np.mean(recalls),
        "HitRate@{}".format(k): np.mean(hit_rates)
    }

In [37]:
# 1. Podział danych
train_df, test_df = train_test_split_per_user(df)

# 2. Trening RP3β tylko na train
rp3 = RP3Beta(beta=0.6, top_k=10)
rp3.fit(train_df)

# 3. Ewaluacja
results = evaluate_rp3beta(rp3, train_df, test_df, k=10)

print("Wyniki RP3β:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

Wyniki RP3β:
Precision@10: 0.0585
Recall@10: 0.1926
HitRate@10: 0.4570


# Model NCF


In [25]:
class NCF(nn.Module):
    def __init__(self, num_users, num_wines, embedding_dim=50):
        super(NCF, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.wine_embedding = nn.Embedding(num_wines, embedding_dim)
        
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.wine_embedding.weight, std=0.01)
        
        self.layer1 = nn.Linear(embedding_dim*2, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.layer2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.layer3 = nn.Linear(64, 1)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) 
        
    def forward(self, user, wine):
        user_emb = self.user_embedding(user)
        wine_emb = self.wine_embedding(wine)
        x = torch.cat([user_emb, wine_emb], dim=1)
        
        x = self.relu(self.bn1(self.layer1(x)))
        x = self.dropout(x)
        
        x = self.relu(self.bn2(self.layer2(x)))
        x = self.dropout(x)
        
        x = self.layer3(x)
        return x

model_ncf = NCF(num_users, num_wines, embedding_dim=50)

print(model_ncf)

NCF(
  (user_embedding): Embedding(10357, 50)
  (wine_embedding): Embedding(1000, 50)
  (layer1): Linear(in_features=100, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer3): Linear(in_features=64, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)


In [30]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 2048
train_dataset = TensorDataset(X_user_train, X_wine_train, y_train_t)
test_dataset = TensorDataset(X_user_test, X_wine_test, y_test_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Training


In [31]:
import torch.optim as optim
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model_ncf = model_ncf.to(device)

criterion = nn.MSELoss()

# Niższy learning rate + scheduler
optimizer = optim.Adam(model_ncf.parameters(), lr=0.003, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

epochs = 50
best_loss_ncf = float('inf')
patience = 5  # Większa cierpliwość
counter = 0

for epoch in range(epochs):
    # ===== TRAIN =====
    model_ncf.train()
    train_loss = 0
    
    for user, wine, rating in train_loader:
        user = user.to(device)
        wine = wine.to(device)
        rating = rating.to(device)
        
        optimizer.zero_grad()
        preds = model_ncf(user, wine).squeeze()
        loss = criterion(preds, rating)
        loss.backward()
        
        # Gradient clipping - zapobiega exploding gradients
        torch.nn.utils.clip_grad_norm_(model_ncf.parameters(), max_norm=1.0)
        
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # ===== TEST =====
    model_ncf.eval()
    test_loss = 0
    
    with torch.no_grad():
        for user, wine, rating in test_loader:
            user = user.to(device)
            wine = wine.to(device)
            rating = rating.to(device)
            
            preds = model_ncf(user, wine).squeeze()
            loss = criterion(preds, rating)
            test_loss += loss.item()
    
    test_loss /= len(test_loader)
    
    # Scheduler - zmniejsza learning rate gdy test loss się nie poprawia
    scheduler.step(test_loss)
    
    print(f"Epoch {epoch+1}/{epochs}  Train: {train_loss:.4f}  Test: {test_loss:.4f}")
    
    if test_loss < best_loss_ncf:
        best_loss_ncf = test_loss
        counter = 0
        torch.save(model_ncf.state_dict(), "best_model_ncf.pt")
        print(f"  ✓ New best model saved! (Test Loss: {test_loss:.4f})")
    else:
        counter += 1
    
    if counter >= patience:
        print(f"Early stopping triggered on epoch {epoch+1}")
        break

# Załaduj najlepszy model
model_ncf.load_state_dict(torch.load("best_model_ncf.pt"))
print(f"\n✓ Best NCF model loaded with Test Loss: {best_loss_ncf:.4f}")

Epoch 1/50  Train: 3.2604  Test: 0.3704
  ✓ New best model saved! (Test Loss: 0.3704)
Epoch 2/50  Train: 0.5112  Test: 0.2957
  ✓ New best model saved! (Test Loss: 0.2957)
Epoch 3/50  Train: 0.4064  Test: 0.2380
  ✓ New best model saved! (Test Loss: 0.2380)
Epoch 4/50  Train: 0.3573  Test: 0.2664
Epoch 5/50  Train: 0.3223  Test: 0.2412
Epoch 6/50  Train: 0.2970  Test: 0.2460
Epoch 7/50  Train: 0.2654  Test: 0.2339
  ✓ New best model saved! (Test Loss: 0.2339)
Epoch 8/50  Train: 0.2490  Test: 0.2357
Epoch 9/50  Train: 0.2364  Test: 0.2409
Epoch 10/50  Train: 0.2272  Test: 0.2419
Epoch 11/50  Train: 0.2118  Test: 0.2423
Epoch 12/50  Train: 0.2031  Test: 0.2434
Early stopping triggered on epoch 12

✓ Best NCF model loaded with Test Loss: 0.2339


In [32]:
# Ewaluacja NCF
metrics_ncf, preds_ncf = evaluate_model(
    model_ncf, test_loader, X_test_clean, y_test_clean.values,
    user2_id, wine2_id, device, "NCF"
)


# XGBOOST

In [33]:
class GradientBoostingPyTorch(nn.Module):
    def __init__(self, num_users, num_wines, embedding_dim=32, n_trees=100):
        super(GradientBoostingPyTorch, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.wine_embedding = nn.Embedding(num_wines, embedding_dim)
        
        self.n_trees = n_trees
        self.trees = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embedding_dim*2, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(32, 1)
            ) for _ in range(n_trees)
        ])

        self.learning_rate = 0.05
        
    def forward(self, user, wine):
        user_emb = self.user_embedding(user)
        wine_emb = self.wine_embedding(wine)
        x = torch.cat([user_emb, wine_emb], dim=1)
        
        output = torch.zeros(x.shape[0], 1, device=x.device) # suma predykcji ze wszystkich drzew
        for tree in self.trees:
            output += self.learning_rate * tree(x)
        
        return output

In [34]:
model_gb = GradientBoostingPyTorch(num_users, num_wines, embedding_dim=32, n_trees=50)
print(model_gb)

GradientBoostingPyTorch(
  (user_embedding): Embedding(10357, 32)
  (wine_embedding): Embedding(1000, 32)
  (trees): ModuleList(
    (0-49): 50 x Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=64, out_features=32, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.3, inplace=False)
      (6): Linear(in_features=32, out_features=1, bias=True)
    )
  )
)


In [ ]:
model_gb = model_gb.to(device)
criterion_gb = nn.MSELoss()
optimizer_gb = optim.Adam(model_gb.parameters(), lr=0.001, weight_decay=1e-5)

In [ ]:
epochs_gb = 20
best_loss_gb = float('inf')
patience_gb = 3
counter_gb = 0

for epoch in range(epochs_gb):
    # TRAIN
    model_gb.train()
    train_loss = 0
    for user, wine, rating in train_loader:
        user = user.to(device)
        wine = wine.to(device)
        rating = rating.to(device)
        optimizer_gb.zero_grad()
        preds = model_gb(user, wine).squeeze()
        loss = criterion_gb(preds, rating)
        loss.backward()
        optimizer_gb.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    # TEST
    model_gb.eval()
    test_loss = 0
    with torch.no_grad():
        for user, wine, rating in test_loader:
            user = user.to(device)
            wine = wine.to(device)
            rating = rating.to(device)
            preds = model_gb(user, wine).squeeze()
            loss = criterion_gb(preds, rating)
            test_loss += loss.item()
    test_loss /= len(test_loader)
    
    print(f"Epoch {epoch+1}/{epochs_gb}  Train: {train_loss:.4f}  Test: {test_loss:.4f}")

metrics_gb, preds_gb = evaluate_model(
    model_gb, test_loader, X_test_clean, y_test_clean.values,
    user2_id, wine2_id, device, "Gradient Boosting"
)

Epoch 1/20  Train: 3.8906  Test: 0.4558
Epoch 2/20  Train: 0.4575  Test: 0.3497
Epoch 3/20  Train: 0.3918  Test: 0.3147
Epoch 4/20  Train: 0.3536  Test: 0.2862
Epoch 5/20  Train: 0.3298  Test: 0.2754
Epoch 6/20  Train: 0.3149  Test: 0.2675
Epoch 7/20  Train: 0.3051  Test: 0.2601
Epoch 8/20  Train: 0.2990  Test: 0.2578
Epoch 9/20  Train: 0.2921  Test: 0.2570
Epoch 10/20  Train: 0.2888  Test: 0.2514
Epoch 11/20  Train: 0.2851  Test: 0.2496
Epoch 12/20  Train: 0.2826  Test: 0.2512
Epoch 13/20  Train: 0.2794  Test: 0.2496
Epoch 14/20  Train: 0.2757  Test: 0.2461
Epoch 15/20  Train: 0.2726  Test: 0.2426
Epoch 16/20  Train: 0.2699  Test: 0.2433
Epoch 17/20  Train: 0.2666  Test: 0.2425
Epoch 18/20  Train: 0.2634  Test: 0.2419
Epoch 19/20  Train: 0.2590  Test: 0.2441
Epoch 20/20  Train: 0.2553  Test: 0.2361


# Random Forest

In [37]:
class RandomForestPyTorch(nn.Module):
    def __init__(self, num_users, num_wines, embedding_dim=32, n_estimators=50):
        super(RandomForestPyTorch, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.wine_embedding = nn.Embedding(num_wines, embedding_dim)
        
        self.n_estimators = n_estimators
        self.trees = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embedding_dim*2, 128),
                nn.ReLU(),
                nn.Dropout(0.5),  
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.4),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(32, 1)
            ) for _ in range(n_estimators)
        ])
        
    def forward(self, user, wine, training=True):
        user_emb = self.user_embedding(user)
        wine_emb = self.wine_embedding(wine)
        x = torch.cat([user_emb, wine_emb], dim=1)
        
        outputs = []
        for tree in self.trees:
            if training:
                tree.train()
            else:
                tree.eval()
            outputs.append(tree(x))

        output = torch.stack(outputs).mean(dim=0)
        return output


In [38]:
model_rf = RandomForestPyTorch(num_users, num_wines, embedding_dim=32, n_estimators=30)
print(model_rf)

model_rf = model_rf.to(device)
criterion_rf = nn.MSELoss()
optimizer_rf = optim.Adam(model_rf.parameters(), lr=0.001, weight_decay=1e-5)

RandomForestPyTorch(
  (user_embedding): Embedding(10357, 32)
  (wine_embedding): Embedding(1000, 32)
  (trees): ModuleList(
    (0-29): 30 x Sequential(
      (0): Linear(in_features=64, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.5, inplace=False)
      (3): Linear(in_features=128, out_features=64, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.4, inplace=False)
      (6): Linear(in_features=64, out_features=32, bias=True)
      (7): ReLU()
      (8): Dropout(p=0.3, inplace=False)
      (9): Linear(in_features=32, out_features=1, bias=True)
    )
  )
)


In [39]:
epochs_rf = 20
best_loss_rf = float('inf')
patience_rf = 3
counter_rf = 0

for epoch in range(epochs_rf):
    # TRAIN
    model_rf.train()
    train_loss = 0
    for user, wine, rating in train_loader:
        user = user.to(device)
        wine = wine.to(device)
        rating = rating.to(device)
        optimizer_rf.zero_grad()
        preds = model_rf(user, wine, training=True).squeeze()
        loss = criterion_rf(preds, rating)
        loss.backward()
        optimizer_rf.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    model_rf.eval()
    test_loss = 0
    with torch.no_grad():
        for user, wine, rating in test_loader:
            user = user.to(device)
            wine = wine.to(device)
            rating = rating.to(device)
            preds = model_rf(user, wine, training=False).squeeze()
            loss = criterion_rf(preds, rating)
            test_loss += loss.item()
    test_loss /= len(test_loader)
    
    print(f"Epoch {epoch+1}/{epochs_rf}  Train: {train_loss:.4f}  Test: {test_loss:.4f}")
    
    if test_loss < best_loss_rf:
        best_loss_rf = test_loss
        counter_rf = 0
        torch.save(model_rf.state_dict(), "best_model_rf.pt")
    else:
        counter_rf += 1
    
    if counter_rf >= patience_rf:
        print("Early stopping triggered on epoch", epoch+1)
        break

rmse_rf = torch.sqrt(torch.tensor(best_loss_rf))
print(f"\n✓ Random Forest (PyTorch) Best RMSE: {rmse_rf.item():.6f}")

metrics_rf, preds_rf = evaluate_model(
    model_rf, test_loader, X_test_clean, y_test_clean.values,
    user2_id, wine2_id, device, "Random Forest"
)

Epoch 1/20  Train: 4.5738  Test: 0.4904
Epoch 2/20  Train: 0.4662  Test: 0.3535
Epoch 3/20  Train: 0.4059  Test: 0.3184
Epoch 4/20  Train: 0.3727  Test: 0.2946
Epoch 5/20  Train: 0.3507  Test: 0.2775
Epoch 6/20  Train: 0.3360  Test: 0.2744
Epoch 7/20  Train: 0.3286  Test: 0.2667
Epoch 8/20  Train: 0.3209  Test: 0.2608
Epoch 9/20  Train: 0.3165  Test: 0.2577
Epoch 10/20  Train: 0.3116  Test: 0.2624
Epoch 11/20  Train: 0.3093  Test: 0.2591
Epoch 12/20  Train: 0.3027  Test: 0.2598
Early stopping triggered on epoch 12

✓ Random Forest (PyTorch) Best RMSE: 0.507689


In [1]:
# PODSUMOWANIE WYNIKÓW - SKRÓCONE

print("SZCZEGÓŁOWE PODSUMOWANIE WYNIKÓW")

# Tworzymy DataFrame tylko z wybranych metryk
results_df = pd.DataFrame([metrics_ncf, metrics_gb, metrics_rf])
results_df = results_df.set_index('Model')
results_df = results_df[['RMSE', 'R2', 'Precision@10']]

print("\n" + results_df.to_string())
print("\n" + "="*60)

# Najlepszy model według wybranych metryk
print("\nNajlepsze modele według różnych metryk:")
print("-" * 60)
for metric in ['RMSE', 'R2', 'Precision@10']:
    if metric == 'R2':
        best_model = results_df[metric].idxmax()
        best_value = results_df[metric].max()
    else:
        best_model = results_df[metric].idxmin() if metric == 'RMSE' else results_df[metric].idxmax()
        best_value = results_df[metric].min() if metric == 'RMSE' else results_df[metric].max()
    print(f"{metric:12s}: {best_model:20s} ({best_value:.4f})")

print("="*60)

# Wizualizacja wyników (tylko RMSE, R2, Precision@10)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Porównanie modeli - wybrane metryki', fontsize=16, fontweight='bold')

metrics_to_plot = ['RMSE', 'R2', 'Precision@10']
for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    
    values = results_df[metric].values
    models = results_df.index.values
    
    bars = ax.bar(models, values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=45)
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=9)
    
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
print("\nWykres zapisany jako 'model_comparison.png'")
plt.show()

# Zapis wyników
results_df.to_csv('model_metrics.csv')
print("Wyniki zapisane do 'model_metrics.csv'")


SZCZEGÓŁOWE PODSUMOWANIE WYNIKÓW


NameError: name 'pd' is not defined